In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

os.getenv("OPENAI_BASE_URL")
os.getenv("OPENAI_MODEL")


'google/gemma-4-31B-it'

In [ ]:
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

def llm(prompt: str) -> str:
    """
    Function to call the LLM and return the response.
    Using the chat completion endpoint because informaniak's provider does not support the responses endpoint yet.
    """
    response = openai_client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL"),
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


llm("Am I Slim Shady?")

"That depends. Do you have a tendency to rhyme complex multisyllabic words, a complicated relationship with your mother, and a desire to stir up controversy in the early 2000s?\n\nIf you are actually **Marshall Mathers**, then yes.\n\nOtherwise, you're probably just a fan. But just to be safe: **do you feel the urge to start a rap battle with everyone in the room?**"

In [21]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

question = "I just discovered the course. Can I join now?"


prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

llm(prompt)


'Yes, you can still join. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted.'

In [40]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [27]:
import requests

documents = []
docs_url = "https://datatalks.club/faq/json/courses.json"

response = requests.get(docs_url)

courses = response.json()

url_prefix = "https://datatalks.club/faq"
for course in courses:
    course_url = f"{url_prefix}/{course['path']}"

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)


1400

In [35]:
from minsearch import Index

index = Index(
    text_fields = ["question", "answer", "section"],
    keyword_fields = ["course"]
)

index.fit(documents)


In [38]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )


search_results = search(question)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [48]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()



def build_prompt(question, search_results):
    context = build_context(search_results)

    prompt = USER_PROMPT_TEMPLATE.format(
        question=question, 
        context=context
    )

    return prompt.strip()


def llm(prompt: str) -> str:
    """
    Function to call the LLM and return the response.
    Using the chat completion endpoint because informaniak's provider does not support the responses endpoint yet.
    """
    response = openai_client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL"),
        messages=[
            {"role": "developer", "content": INSTRUCTIONS},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content



In [56]:
# prompt = build_prompt(question, search_results)

# llm(prompt)

print(rag("How are you?"))
print(rag(question))
print(rag("When is the next cohort?"))

I don't know.
Yes, you can still join. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted.
The course will be offered next in Summer 2027.
